# EDA Notebook 3 — Video Watch Events Bronze
**Source**: `mini_project_grp6.bronze.video_watch_bronze`  
**Format**: Parquet | ~8M rows/day | completion_pct, pause/seek counts  
**Goal**: Completion rates, engagement depth, device patterns, buffering issues

In [0]:
from pyspark.sql import functions as F

TABLE = "mini_project_grp6.bronze.video_watch_bronze"
df    = spark.table(TABLE)
total = df.count()

print(f"Total rows: {total:,}")
df.printSchema()

In [0]:
# ============================================================
# CELL 2 — Null Audit
# ============================================================

null_df = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).toPandas().T.reset_index()
null_df.columns = ["column", "null_count"]
null_df["null_pct"] = (null_df["null_count"] / total * 100).round(2)
print(null_df.sort_values("null_count", ascending=False).to_string(index=False))

In [0]:
# ============================================================
# CELL 3 — Completion % Distribution
# ============================================================

print("=== COMPLETION PCT STATS ===")
df.select("completion_pct").describe().show()

print("\n=== COMPLETION BUCKETS ===")
display(
    df.withColumn("completion_bucket",
        F.when(F.col("completion_pct") == 0, "0% (Not started)")
         .when(F.col("completion_pct") < 25, "1-24%")
         .when(F.col("completion_pct") < 50, "25-49%")
         .when(F.col("completion_pct") < 75, "50-74%")
         .when(F.col("completion_pct") < 100, "75-99%")
         .otherwise("100% (Complete)")
    )
    .groupBy("completion_bucket")
    .agg(
        F.count("*").alias("count"),
        F.round(F.count("*") / total * 100, 2).alias("pct")
    )
    .orderBy("completion_bucket")
)

In [0]:
# ============================================================
# CELL 4 — Video Engagement Metrics
# ============================================================
# watched_seconds vs video_duration_seconds

df.selectExpr(
    "percentile_approx(video_duration_seconds, 0.5)  as median_video_len_s",
    "percentile_approx(watched_seconds, 0.5)          as median_watched_s",
    "avg(pause_count)                                  as avg_pauses",
    "avg(seek_count)                                   as avg_seeks",
    "avg(buffering_events)                             as avg_buffering"
).show()

In [0]:
# ============================================================
# CELL 5 — Playback Speed Distribution
# ============================================================

display(
    df.groupBy("playback_speed")
      .agg(F.count("*").alias("count"))
      .orderBy("playback_speed")
)

In [0]:
# ============================================================
# CELL 6 — Device Type in Video Watching
# ============================================================

display(
    df.groupBy("device_type")
      .agg(
          F.count("*").alias("count"),
          F.round(F.avg("completion_pct"), 2).alias("avg_completion_pct"),
          F.round(F.avg("buffering_events"), 2).alias("avg_buffering")
      )
      .orderBy("count", ascending=False)
)

In [0]:
# ============================================================
# CELL 7 — Logical Check: watched_seconds > video_duration
# ============================================================

over_watched = df.filter(
    F.col("watched_seconds") > F.col("video_duration_seconds")
)
print(f"Records where watched > duration: {over_watched.count():,}")
display(over_watched.select(
    "watch_id", "video_duration_seconds", "watched_seconds", "completion_pct"
).limit(10))

In [0]:
# ============================================================
# CELL 8 — Watch Date Coverage
# ============================================================

df.selectExpr(
    "min(watch_date) as earliest",
    "max(watch_date) as latest",
    "count(distinct watch_date) as distinct_days"
).show()

display(
    df.groupBy("watch_date")
      .count()
      .orderBy("watch_date")
)